In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pickle
import warnings

In [ ]:
# Ignore harmless warnings from statsmodels
warnings.filterwarnings("ignore")

In [ ]:
# ==========================================
# 1. Read and preprocess the data
# ==========================================
print("Reading data...")
try:
    df = pd.read_csv('../data/BTCUSD1D.csv')
except FileNotFoundError:
    print("Error: File 'BTCUSD1D.csv' not found.")
    exit()

In [ ]:
# Convert datetime column to datetime format and sort chronologically
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime')
df = df.set_index('datetime')

In [ ]:
# We only need the 'close' column for standard SARIMA
data = df['close']

In [ ]:
# ==========================================
# 2. Split Train / Test sets
# ==========================================
split_date = '2026-03-01'

In [ ]:
train_data = data.loc[data.index < split_date]
test_data = data.loc[data.index >= split_date]

In [ ]:
print(f"Number of Train samples: {len(train_data)}")
print(f"Number of Test samples: {len(test_data)}")

In [ ]:
if len(train_data) == 0 or len(test_data) == 0:
    print("Error: Not enough data for Train/Test split.")
    exit()

In [ ]:
# ==========================================
# 3. Train the SARIMA model on Training Data
# ==========================================
print("Training SARIMA model on historical data...")
order = (1, 1, 1) # p=1, d=1, q=1
seasonal_order = (0, 0, 0, 0)

In [ ]:
# Fit model ONLY on train_data to learn the patterns
train_model = SARIMAX(train_data, order=order, seasonal_order=seasonal_order, enforce_stationarity=False, enforce_invertibility=False)
fitted_train_model = train_model.fit(disp=False)

In [ ]:
# ==========================================
# 4. Apply learned parameters for Rolling Forecast
# ==========================================
print("Applying learned parameters to the full dataset for 1-step ahead prediction...")
# Create a new model structure using the FULL dataset
full_model = SARIMAX(data, order=order, seasonal_order=seasonal_order, enforce_stationarity=False, enforce_invertibility=False)

In [ ]:
# Apply (filter) the weights learned from the training set to this new model
# This prevents data leakage (we don't train on test data) but allows using actual past values
rolling_model = full_model.filter(fitted_train_model.params)

In [ ]:
# Predict specifically for the test period
# dynamic=False ensures it uses the TRUE value of t-1 to predict t
predictions = rolling_model.predict(start=split_date, end=data.index[-1], dynamic=False)

In [ ]:
# Create a DataFrame for results
results_df = pd.DataFrame({
    'Actual Price': test_data,
    'Predicted Price': predictions
})

In [ ]:
# ==========================================
# 5. Evaluate the model
# ==========================================
mae = mean_absolute_error(results_df['Actual Price'], results_df['Predicted Price'])
mse = mean_squared_error(results_df['Actual Price'], results_df['Predicted Price'])
print(f"MAE on test set: {mae:.2f}")
print(f"MSE on test set: {mse:.2f}")

In [ ]:
print("\nPrediction results (using actual previous day's price) from 2026-03-01 onwards:")
print(results_df.round(2).head(20))

In [ ]:
# ==========================================
# 6. Visualize predicted and actual prices
# ==========================================
plt.figure(figsize=(12, 6))
plt.plot(results_df.index, results_df['Actual Price'], label='Actual Price', color='blue', alpha=0.7)
plt.plot(results_df.index, results_df['Predicted Price'], label='Predicted Price', color='red', linestyle='--', alpha=0.7)
plt.title('BTC Price Prediction (SARIMA 1-Step Rolling Forecast)')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ==========================================
# 7. Save the model for reuse
# ==========================================
model_filename = 'models/2_sarima_rolling_btc_model.pkl'
with open(model_filename, 'wb') as f:
    # Save the fitted training model parameters
    pickle.dump(fitted_train_model, f)
print(f"\nModel successfully saved to file: '{model_filename}'")

In [ ]:
# ==========================================
# 8. Example: Predict the next future session
# ==========================================
# rolling_model already has the entire actual dataset loaded up to today
next_session_pred = rolling_model.forecast(steps=1)
print(f"\n--> Predicted BTC price for the next upcoming session is: {next_session_pred.iloc[0]:.2f}")